# SimCLR Implementation

Training hyperparameters follow the paper (Appendix F.1, Van Gansbeke et al. 2020):
- **Architecture:** ResNet-18 encoder + MLP projection head (512 → 128)
- **Optimizer:** SGD, 0.9 momentum, weight decay 0.0001
- **Learning rate:** 0.4, cosine scheduler
- **Batch size:** 512
- **Epochs:** 500
- **Augmentations:** Random resized crop, horizontal flip, color jitter, random grayscale
- **Loss:** NT-Xent (τ=0.5)
- **Embedding:** L2-normalized 512-d penultimate layer

In [11]:
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torchvision
import torchvision.models as models

from torch.utils.data import DataLoader
from tqdm import tqdm

### Augmentation Transform

In [12]:
class SimCLRTransform:
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size=32),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)

### ResNet-18 Encoder

In [13]:

class ResNet18Encoder(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18Encoder, self).__init__()
        self.resnet = models.resnet18(pretrained=False)
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.resnet.maxpool = nn.Identity()  # Remove maxpool layer for CIFAR-10
        self.resnet.fc = nn.Identity()  # Remove fully connected layer for feature extraction

    def forward(self, x):
        return self.resnet(x)

### Projection Head

In [14]:
class ProjectionHead(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128, output_dim=128):
        super(ProjectionHead, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

### NT_Xent Loss

In [15]:
def nt_xent_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.size(0)
    z = torch.cat([z_i, z_j], dim=0)  # Concatenate positive pairs
    
    z = nn.functional.normalize(z, dim=1)  # L2 Norm
    similarity_matrix = torch.matmul(z, z.T)/temperature  # Compute similarity matrix

    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
    similarity_matrix = similarity_matrix[~mask].view(2 * batch_size, -1)  # Remove self-similarity

    targets = torch.cat([
        torch.arange(batch_size -1, 2*batch_size -1),
        torch.arange(0, batch_size)
    ]).to(z.device)

    loss = nn.CrossEntropyLoss()(similarity_matrix, targets)
    return loss

### Import Dataset

In [ ]:
simclr_trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                                download=True, transform=SimCLRTransform())
simclr_loader = DataLoader(simclr_trainset, batch_size=512, shuffle=True, num_workers=2, drop_last=True)

### Training Loop Setup

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Initialize models
encoder = ResNet18Encoder().to(device)
projection_head = ProjectionHead().to(device)

# Optimizer — matches paper: SGD, 0.9 momentum, lr=0.4, cosine schedule, weight decay 0.0001
num_epochs = 500
optimizer = torch.optim.SGD(
    list(encoder.parameters()) + list(projection_head.parameters()),
    lr=0.4, momentum=0.9, weight_decay=0.0001
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print(f'Training for {num_epochs} epochs')
print(f'Optimizer: SGD (lr=0.4, momentum=0.9, weight_decay=0.0001)')
print(f'Scheduler: CosineAnnealingLR')

### Training Loop

In [ ]:
for epoch in range(num_epochs):
    running_loss = 0.0
    loop = tqdm(simclr_loader, desc=f'Epoch [{epoch+1}/{num_epochs}]', leave=False)
    for (view1, view2), _ in loop:
        view1, view2 = view1.to(device), view2.to(device)

        h_i = encoder(view1)
        h_j = encoder(view2)
        z_i = projection_head(h_i)
        z_j = projection_head(h_j)

        loss = nt_xent_loss(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    scheduler.step()
    avg_loss = running_loss / len(simclr_loader)
    if (epoch + 1) % 50 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Avg Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}')

# Save encoder weights
torch.save(encoder.state_dict(), 'simclr_encoder_500ep.pth')
print('Training complete. Encoder saved to simclr_encoder_500ep.pth')